<a href="https://colab.research.google.com/github/lakshaykumar11/tts-quantization-indic/blob/main/tts_quant_02_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TTS Quantization — Pilot

Runtime → T4 GPU. Upload `testset.csv`. Then Run all **twice** (cell 2 restarts the session on purpose).

In [ ]:
!pip install -q coqui-tts
!pip install -q -U "transformers<5"
!pip install -q -U openai-whisper
!pip install -q jiwer indic-transliteration

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

if not os.path.exists('/content/.restarted'):
    open('/content/.restarted', 'w').close()
    os.kill(os.getpid(), 9)
print("ready")

In [ ]:
import os, re, time, glob
import pandas as pd, jiwer
from TTS.api import TTS
import whisper

OUT = '/content/tts_quant'
os.makedirs(f'{OUT}/audio', exist_ok=True)
os.makedirs(f'{OUT}/results', exist_ok=True)

SPEAKER = "Ana Florence"

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
asr = whisper.load_model("large-v3")
print("models loaded")

In [ ]:
df = pd.read_csv('testset (3).csv')
print(df.shape)
print(df.groupby('condition').size())

## Scoring

Whisper transcribes to Devanagari regardless of input script, so romanized references need a script-neutral comparison. Both sides are collapsed to a coarse phonetic form before scoring.

In [ ]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

DEVA = re.compile(r'[\u0900-\u097F]')
CONS = 'bcdfghjklmnpqrstvwxyz'


def _tokens(text):
    out = []
    for tok in text.split():
        if DEVA.search(tok):
            out.append((transliterate(tok, sanscript.DEVANAGARI, sanscript.ITRANS), True))
        else:
            out.append((tok, False))
    return out


def _norm_word(w, from_deva):
    if from_deva:
        w = w.replace('A', 'aa').replace('I', 'ii').replace('U', 'uu')
        w = w.replace('M', 'n').replace('~', 'n').replace('.', '')
    w = re.sub(r'[^a-z]', '', w.lower())
    if not w:
        return ''
    if from_deva:
        w = re.sub(r'(?<!a)a$', '', w)
        prev = None
        while prev != w:
            prev = w
            w = re.sub(r'([aeiou][%s])a([%s][aeiou])' % (CONS, CONS), r'\1\2', w)
    w = w.replace('w', 'v').replace('y', 'i')
    w = w.replace('aa', 'a').replace('ii', 'i').replace('uu', 'u')
    w = w.replace('ee', 'i').replace('oo', 'u')
    w = w.replace('ai', 'e').replace('au', 'o')
    w = w.replace('c', 'k').replace('q', 'k').replace('x', 'ks')
    w = re.sub(r'([%s])h' % CONS, r'\1', w)
    return re.sub(r'(.)\1+', r'\1', w)


def phonetic_norm(text):
    return ' '.join(x for x in (_norm_word(t, d) for t, d in _tokens(text)) if x)


def plain_norm(text):
    text = re.sub(r"[।.,!?;:\"'\-]", "", text.lower())
    return re.sub(r"\s+", " ", text).strip()


def synth(text, lang, path):
    t0 = time.time()
    tts.tts_to_file(text=text, speaker=SPEAKER, language=lang, file_path=path)
    return time.time() - t0


def score(path, reference, asr_lang, neutral=False):
    heard = asr.transcribe(path, language=asr_lang)["text"]
    f = phonetic_norm if neutral else plain_norm
    ref, hyp = f(reference), f(heard)
    return {"heard": heard.strip(), "wer": jiwer.wer(ref, hyp), "cer": jiwer.cer(ref, hyp)}

In [ ]:
_pairs = [
    ("Maine kal raat apna project poora kar liya tha",
     "मैंने कल रात अपना प्रोजेक्ट पूरा कर लिया था"),
    ("Kal ki meeting cancel ho gayi isliye main ghar raha",
     "कल की मीटिंग कैंसल हो गई इसलिए मैं घर रहा"),
    ("Mujhe kal jaldi office jaakar morning meeting attend karni hai",
     "मुझे कल जल्दी ऑफिस जाकर मॉर्निंग मीटिंग अटेंड करनी है"),
]

print("noise floor:\n")
for lat, dev in _pairs:
    a, b = phonetic_norm(lat), phonetic_norm(dev)
    print(f"  {a}\n  {b}\n  CER {jiwer.cer(a, b):.3f}\n")

## Sanity check

In [ ]:
for cond, lang in [("EN", "en"), ("HI", "hi")]:
    row = df[df.condition == cond].iloc[0]
    p = f"{OUT}/audio/sanity_{row.id}.wav"
    secs = synth(row.text, lang, p)
    r = score(p, row.text, lang)
    print(f"[{cond}] WER {r['wer']:.2f} | CER {r['cer']:.2f} | {secs:.0f}s")
    print(f"   said : {row.text}")
    print(f"   heard: {r['heard']}\n")

## Pilot — language tag for Hinglish

XTTS requires a `language` argument but code-mixed text has no correct value. Both options are run on the same sentences.

In [ ]:
pilot = []

for cond in ["HING_MIX", "HING_ROM"]:
    sub = df[df.condition == cond]
    for lvl in ["low", "medium", "high"]:
        row = sub[sub.mix_level == lvl].iloc[0]
        for tag in ["hi", "en"]:
            p = f"{OUT}/audio/pilot_{row.id}_{tag}.wav"
            secs = synth(row.text, tag, p)
            r = score(p, row.text, "hi", neutral=True)
            pilot.append({"id": row.id, "condition": cond, "mix_level": lvl,
                          "cmi": row.cmi, "tag": tag, "secs": round(secs, 1),
                          "wer": round(r["wer"], 3), "cer": round(r["cer"], 3),
                          "text": row.text, "heard": r["heard"]})
            print(f"{row.id} [{tag}] WER {r['wer']:.2f} CER {r['cer']:.2f}")

pilot_df = pd.DataFrame(pilot)
pilot_df.to_csv(f"{OUT}/results/pilot.csv", index=False)
pilot_df[["id", "condition", "mix_level", "cmi", "tag", "wer", "cer"]]

In [ ]:
print(pilot_df.groupby(["condition", "tag"])[["wer", "cer"]].mean().round(3), "\n")
print(pilot_df[pilot_df.tag == "hi"].groupby(["condition", "mix_level"])[["cmi", "wer", "cer"]].mean().round(3))

In [ ]:
from IPython.display import Audio, display

for f in sorted(glob.glob(f'{OUT}/audio/*.wav')):
    print(os.path.basename(f))
    display(Audio(f))

In [ ]:
from google.colab import files
files.download(f'{OUT}/results/pilot.csv')